# 25 — Adaptive RAG

Route each query to the **cheapest strategy** that correctly answers it: direct LLM, private vectorstore, or live web search.

**What you'll learn**
- Why a single retrieval strategy applied to every query wastes money and hurts quality
- How an LLM classifier picks the right path before any retrieval happens
- Three strategies: `simple` (LLM only), `vectorstore` (private KB), `web` (live DuckDuckGo)
- Using `with_structured_output` for a reliable `Literal`-typed routing decision
- The `classify → route → answer` pattern with conditional edges

**Companion examples:** 17-corrective-rag (grades docs AFTER retrieval), 27-self-rag (decides WHETHER to retrieve), 26-rag-fusion (better recall via query expansion)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langchain-chroma langchain-community chromadb duckduckgo-search python-dotenv langgraph pydantic

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Private knowledge base ─────────────────────────────────────────────────
# These represent private/internal docs the LLM doesn't know.
# The classifier routes policy and spec questions here.
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

DOCS = [
    Document(page_content="Our refund policy allows returns within 30 days of purchase with original receipt."),
    Document(page_content="API rate limits are 1000 requests per minute for Pro plans and 100 for Free plans."),
    Document(page_content="The platform supports SSO via SAML 2.0 and OAuth 2.0 providers including Google and GitHub."),
    Document(page_content="Data is stored in EU-West-1 by default. US regions available on Enterprise plans."),
    Document(page_content="Uptime SLA is 99.9% for Pro plans and 99.99% for Enterprise. Includes status page monitoring."),
]

vectorstore = Chroma.from_documents(
    DOCS, OpenAIEmbeddings(model="text-embedding-3-small"), collection_name="adaptive_rag"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Private KB ready with {len(DOCS)} documents")

In [ ]:
# ── 4. Build the adaptive routing graph ───────────────────────────────────────
# RouteDecision is the structured output the classifier returns.
# The route() function maps the strategy string to the correct answer node.
from typing import Literal, TypedDict

from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel

Strategy = Literal["simple", "vectorstore", "web"]


class RouteDecision(BaseModel):
    strategy: Strategy
    reasoning: str


class AdaptiveRAGState(TypedDict):
    question: str
    strategy: str
    context: str
    answer: str


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
search = DuckDuckGoSearchResults(max_results=3)
classifier = llm.with_structured_output(RouteDecision)


def classify(state: AdaptiveRAGState) -> dict:
    """Choose the cheapest strategy that will answer correctly."""
    decision = classifier.invoke([
        SystemMessage(
            "Classify this question into one of three retrieval strategies:\n"
            "- 'simple': general knowledge the LLM knows well (math, history, definitions)\n"
            "- 'vectorstore': private/internal knowledge (policies, specs, internal docs)\n"
            "- 'web': current or real-time information (news, prices, recent events)\n"
            "Choose the cheapest strategy that answers correctly."
        ),
        HumanMessage(state["question"]),
    ])
    return {"strategy": decision.strategy}


def direct_answer(state: AdaptiveRAGState) -> dict:
    response = llm.invoke([SystemMessage("Answer concisely."), HumanMessage(state["question"])])
    return {"context": "none", "answer": response.content}


def vectorstore_answer(state: AdaptiveRAGState) -> dict:
    docs = retriever.invoke(state["question"])
    context = "\n\n".join(d.page_content for d in docs)
    response = llm.invoke([
        SystemMessage(f"Answer using only:\n{context}"),
        HumanMessage(state["question"]),
    ])
    return {"context": context, "answer": response.content}


def web_answer(state: AdaptiveRAGState) -> dict:
    results = search.invoke(state["question"])
    response = llm.invoke([
        SystemMessage(f"Answer using these web results:\n{results}"),
        HumanMessage(state["question"]),
    ])
    return {"context": str(results), "answer": response.content}


def route(
    state: AdaptiveRAGState,
) -> Literal["direct_answer", "vectorstore_answer", "web_answer"]:
    return f"{state['strategy']}_answer"


graph = StateGraph(AdaptiveRAGState)
graph.add_node("classify", classify)
graph.add_node("direct_answer", direct_answer)
graph.add_node("vectorstore_answer", vectorstore_answer)
graph.add_node("web_answer", web_answer)
graph.add_edge(START, "classify")
graph.add_conditional_edges(
    "classify",
    route,
    {"direct_answer": "direct_answer", "vectorstore_answer": "vectorstore_answer", "web_answer": "web_answer"},
)
graph.add_edge("direct_answer", END)
graph.add_edge("vectorstore_answer", END)
graph.add_edge("web_answer", END)
app = graph.compile()

print("Graph: START → classify → [direct_answer | vectorstore_answer | web_answer] → END")

In [ ]:
# ── 5. Run the routing demo ───────────────────────────────────────────────────
# Expected routes are annotated in comments. Check how well the classifier does.
SAMPLE_QUESTIONS = [
    "What is 7 multiplied by 8?",                    # simple      — LLM knows math
    "What is your refund policy?",                   # vectorstore — private policy
    "What are the API rate limits for a Pro plan?",  # vectorstore — private spec
    "What is the current price of Bitcoin?",         # web         — real-time data
    "Who wrote Hamlet?",                             # simple      — general knowledge
]

for question in SAMPLE_QUESTIONS:
    result = app.invoke({"question": question, "strategy": "", "context": "", "answer": ""})
    print(f"Q: {question}")
    print(f"  Strategy : {result['strategy']}")
    print(f"  Answer   : {result['answer'][:120]}")
    print()

## Exercises

**Exercise 1 — Inspect the reasoning:** `RouteDecision` has a `reasoning` field. Modify `classify()` to return it in state, then print it for each question. Does the reasoning match your expectation?

**Exercise 2 — Add private docs:** Add 2-3 new documents to `DOCS` (e.g., about your own product). Write questions that should route to `vectorstore`. Verify they land correctly.

**Exercise 3 — Force a misroute:** Change a question to be ambiguous (e.g., "What is our uptime guarantee?" where "our" could mean anything). Watch which strategy the classifier picks and why.

## What's next

- **17-corrective-rag** — grades retrieved docs AFTER retrieval, rewrites query if any score irrelevant
- **27-self-rag** — three LLM gates: classify (retrieve?), grade (relevant?), check (grounded?)
- **26-rag-fusion** — fan out N query variants in parallel, merge with Reciprocal Rank Fusion